In [9]:
import pandas as pd 
import numpy as np 

In [10]:
df = pd.read_csv("customer_support_tickets_200k.csv")

In [11]:
df[(df["priority"] == "Urgent") & (df["sla_breached"] == "No")]

,ticket_id,customer_name,customer_email,product,category,issue_description,resolution_notes,priority,status,channel,...,ticket_resolved_date,escalated,sla_breached,operating_system,browser,payment_method,language,preferred_contact_time,issue_complexity_score,customer_segment
9,10,Elizabeth Miller,elizabeth.miller992@gmail.com,Billing System,Performance Issue,I am unable to access my account after enterin...,We have reset the account credentials and advi...,Urgent,Closed,Social Media,...,2022-05-16,Yes,No,Windows,NaN,Debit Card,Chinese,Morning,10,Individual
23,24,John Wilson,john.wilson181@company.com,Cloud Storage,Feature Request,I found a bug in the latest update affecting r...,Data synchronization restored after backend se...,Urgent,Resolved,Social Media,...,2024-12-14,No,No,Windows,Edge,PayPal,Spanish,Night,4,Small Business
30,31,Joseph Rodriguez,joseph.rodriguez186@hotmail.com,CRM Platform,Subscription Cancellation,The payment was deducted from my bank account ...,Payment gateway timeout issue fixed and monito...,Urgent,Pending Customer,Phone,...,2024-06-21,Yes,No,Windows,Safari,PayPal,French,Morning,7,Corporate
32,33,Patricia Anderson,patricia.anderson970@gmail.com,E-commerce Store,Feature Request,The system is not syncing data across devices ...,Provided step-by-step troubleshooting instruct...,Urgent,Resolved,Chat,...,2022-11-19,Yes,No,iOS,Safari,Crypto,French,Morning,5,Corporate
34,35,John Taylor,john.taylor989@hotmail.com,API Service,Payment Problem,I would like to request a refund for the recen...,The duplicate charge was reversed and refund p...,Urgent,Open,Chat,...,2023-11-24,Yes,No,Linux,Edge,PayPal,Japanese,Night,9,Individual
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199982,199983,Richard Thomas,richard.thomas387@icloud.com,CRM Platform,Security Concern,There seems to be a discrepancy in my billing ...,Provided step-by-step troubleshooting instruct...,Urgent,Resolved,Social Media,...,2023-12-29,Yes,No,Android,Edge,Crypto,French,Morning,7,Corporate
199985,199986,Sarah Miller,sarah.miller819@outlook.com,Cloud Storage,Feature Request,Two-factor authentication codes are not being ...,The duplicate charge was reversed and refund p...,Urgent,Resolved,Chat,...,2023-08-27,No,No,iOS,Edge,Bank Transfer,French,Morning,9,Small Business
199988,199989,James Brown,james.brown737@icloud.com,Cloud Storage,Login Issue,I am unable to access my account after enterin...,Payment gateway timeout issue fixed and monito...,Urgent,Resolved,Social Media,...,2023-09-10,No,No,Android,Edge,Debit Card,English,Morning,10,Small Business
199990,199991,Susan Jones,susan.jones131@hotmail.com,API Service,Payment Problem,Two-factor authentication codes are not being ...,We have reset the account credentials and advi...,Urgent,Open,Web Form,...,2024-05-16,Yes,No,MacOS,NaN,Debit Card,Spanish,Morning,9,Individual


In [12]:


# -----------------------------
# 1. Make results reproducible
# -----------------------------
np.random.seed(42)

# -----------------------------
# 2. Create 25 agents
# -----------------------------
agents = []

agent_names = ["David Okoro","Amina Bello","Tunde Adebayo","Chioma Nwankwo","Ibrahim Musa","Kemi Ogunleye",
    "Emeka Obi",
    "Fatima Abdullahi",
    "Sola Ajayi",
    "Grace Eze",
    "Ahmed Lawal",
    "Ngozi Okafor",
    "Samuel Olatunji",
    "Zainab Sadiq",
    "Chinedu Uche",
    "Maryam Aliyu",
    "Oluwaseun Adeyemi",
    "Blessing Onyekachi",
    "Kabir Usman",
    "Esther Ibe",
    "Daniel Ekanem",
    "Hadiza Mohammed",
    "Uchechukwu Nnamdi",
    "Rukayat Balogun",
    "Peter Essien",
    "Sarah Zoe"]


# Tier 1 agents: 14
for i in range(1, 15):
    agents.append({
        "agent_id": f"A{i:03d}",
        "agent_name": agent_names[i],
        "team": "Tier 1",
        "hire_date": pd.Timestamp("2021-01-01") + pd.to_timedelta(np.random.randint(0, 1400), unit="D")
    })

# Tier 2 agents: 8
for i in range(15, 23):
    agents.append({
        "agent_id": f"A{i:03d}",
        "agent_name": agent_names[i],
        "team": "Tier 2",
        "hire_date": pd.Timestamp("2020-01-01") + pd.to_timedelta(np.random.randint(0, 1800), unit="D")
    })

# Escalation agents: 3
for i in range(23, 26):
    agents.append({
        "agent_id": f"A{i:03d}",
        "agent_name": agent_names[i],
        "team": "Escalations",
        "hire_date": pd.Timestamp("2019-01-01") + pd.to_timedelta(np.random.randint(0, 2200), unit="D")
    })

agents_df = pd.DataFrame(agents)




# -----------------------------
# 4. Define weighted pools
# -----------------------------
tier1_agents = [f"A{i:03d}" for i in range(1, 15)]
tier2_agents = [f"A{i:03d}" for i in range(15, 23)]
escalation_agents = [f"A{i:03d}" for i in range(23, 26)]

tier1_weights = np.array([0.10, 0.09, 0.08, 0.06, 0.09, 0.07, 0.08, 0.07, 0.06, 0.08, 0.07, 0.08, 0.04, 0.03])
tier1_weights = tier1_weights / tier1_weights.sum()

tier2_weights = np.array([0.16, 0.14, 0.12, 0.13, 0.11, 0.12, 0.11, 0.11])
tier2_weights = tier2_weights / tier2_weights.sum()

escalation_weights = np.array([0.36, 0.33, 0.31])
escalation_weights = escalation_weights / escalation_weights.sum()

# -----------------------------
# 5. Clean/standardize columns
# -----------------------------
# Make sure these columns exist and are in usable format
df["escalated"] = df["escalated"].astype(str).str.strip().str.lower()
df["priority"] = df["priority"].astype(str).str.strip().str.lower()
df["channel"] = df["channel"].astype(str).str.strip().str.lower()

# If complexity has missing values, fill with median
df["issue_complexity_score"] = pd.to_numeric(df["issue_complexity_score"], errors="coerce")
df["issue_complexity_score"] = df["issue_complexity_score"].fillna(df["issue_complexity_score"].median())

# -----------------------------
# 6. Assignment function
# -----------------------------
def assign_agent(row):
    escalated = row["escalated"] in ["true", "1", "yes"]
    complexity = row["issue_complexity_score"]
    priority = row["priority"]
    channel = row["channel"]

    # Rule 1: escalated tickets go to escalation team
    if escalated:
        return np.random.choice(escalation_agents, p=escalation_weights)

    # Rule 2: very complex or high-priority phone/email tickets lean Tier 2
    if complexity >= 8:
        return np.random.choice(tier2_agents, p=tier2_weights)

    if priority == "high":
        if np.random.rand() < 0.70:
            return np.random.choice(tier2_agents, p=tier2_weights)
        return np.random.choice(tier1_agents, p=tier1_weights)

    # Rule 3: medium complexity split between Tier 1 and Tier 2
    if 5 <= complexity < 8:
        if np.random.rand() < 0.35:
            return np.random.choice(tier2_agents, p=tier2_weights)
        return np.random.choice(tier1_agents, p=tier1_weights)

    # Rule 4: chat tickets are more likely Tier 1
    if channel == "chat":
        if np.random.rand() < 0.90:
            return np.random.choice(tier1_agents, p=tier1_weights)
        return np.random.choice(tier2_agents, p=tier2_weights)

    # Default
    return np.random.choice(tier1_agents, p=tier1_weights)

# -----------------------------
# 7. Assign agents
# -----------------------------
df["agent_id"] = df.apply(assign_agent, axis=1)

# -----------------------------
# 8. Merge agent metadata into ticket table
# -----------------------------
df = df.merge(
    agents_df[["agent_id", "agent_name", "team", "hire_date"]],
    on="agent_id",
    how="left"
)


# Example SLA thresholds by priority
def sla_threshold(priority):
    if priority == "Urgent":
        return 6
    if priority == "High":
        return 8
    elif priority == "Medium":
        return 24
    else:
        return 48

df["sla_target_hours"] = df["priority"].apply(sla_threshold)
df["sla_breached_recalc"] = df["resolution_time_hours"] > df["sla_target_hours"]


print("\nAgents table:")
print(agents_df.head())


Agents table:
  agent_id      agent_name    team  hire_date
0     A001     Amina Bello  Tier 1 2024-02-01
1     A002   Tunde Adebayo  Tier 1 2023-05-11
2     A003  Chioma Nwankwo  Tier 1 2024-07-18
3     A004    Ibrahim Musa  Tier 1 2024-02-05
4     A005   Kemi Ogunleye  Tier 1 2024-01-01


In [13]:
df.head()

,ticket_id,customer_name,customer_email,product,category,issue_description,resolution_notes,priority,status,channel,...,language,preferred_contact_time,issue_complexity_score,customer_segment,agent_id,agent_name,team,hire_date,sla_target_hours,sla_breached_recalc
0,1,Patricia Smith,patricia.smith760@outlook.com,Web Portal,Account Suspension,The payment was deducted from my bank account ...,Data synchronization restored after backend se...,urgent,Open,email,...,French,Afternoon,4,Small Business,A004,Ibrahim Musa,Tier 1,2024-02-05,48,True
1,2,Patricia Williams,patricia.williams390@gmail.com,Mobile App,Performance Issue,I found a bug in the latest update affecting r...,Provided step-by-step troubleshooting instruct...,urgent,Closed,email,...,English,Afternoon,2,Small Business,A024,Peter Essien,Escalations,2021-08-13,48,True
2,3,William Anderson,william.anderson651@outlook.com,Web Portal,Performance Issue,The application crashes whenever I try to uplo...,Provided step-by-step troubleshooting instruct...,medium,Closed,chat,...,French,Morning,4,Corporate,A024,Peter Essien,Escalations,2021-08-13,48,True
3,4,David Miller,david.miller672@icloud.com,Payment Gateway,Subscription Cancellation,My subscription was cancelled without my reque...,Provided step-by-step troubleshooting instruct...,medium,Closed,social media,...,Spanish,Afternoon,7,Corporate,A023,Rukayat Balogun,Escalations,2022-04-30,48,True
4,5,Robert Gonzalez,robert.gonzalez391@hotmail.com,Web Portal,Feature Request,The system is not syncing data across devices ...,We have reset the account credentials and advi...,high,Pending Customer,email,...,Spanish,Evening,3,Corporate,A024,Peter Essien,Escalations,2021-08-13,48,True


In [14]:
df.drop("sla_breached", inplace= True,axis= 1)

In [15]:
df[df["sla_breached_recalc"] == False].value_counts()

ticket_id  customer_name      customer_email                    product               category                   issue_description                                                                resolution_notes                                                                  priority  status            channel       region         customer_age  customer_gender  subscription_type  customer_tenure_months  previous_tickets  customer_satisfaction_score  first_response_time_hours  resolution_time_hours  ticket_created_date  ticket_resolved_date  escalated  operating_system  browser  payment_method  language  preferred_contact_time  issue_complexity_score  customer_segment  agent_id  agent_name       team         hire_date   sla_target_hours  sla_breached_recalc
9          William Anderson   william.anderson256@gmail.com     API Service           Security Concern           The payment was deducted from my bank account but the transaction shows failed.  We have reset the account credentials a

In [16]:
df.to_csv('Joined_data')